## 📓 Notebook 03: XGBoost Tuning

**Skills practiced:** `max_depth`, `learning_rate` vs `n_estimators`, one-parameter-at-a-time testing, `GridSearchCV`, `roc_auc_score`.

**Goal:** Record a default-parameter baseline, see which hyperparameters move validation ROC-AUC, and find a tuned configuration with a small grid search.

In [84]:
import time
import joblib
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score

In [85]:
y_train=pd.read_csv('../data/processed/y_train.csv')
y_test=pd.read_csv('../data/processed/y_test.csv')
y_val=pd.read_csv('../data/processed/y_val.csv')
x_train=pd.read_csv('../data/processed/train_enc.csv')
x_test=pd.read_csv('../data/processed/test_enc.csv')
x_val=pd.read_csv('../data/processed/validate_enc.csv')

In [86]:
m=xgb.XGBClassifier()
m.fit(x_train, y_train)
base_num=roc_auc_score(y_val, m.predict_proba(x_val)[:,1])
print(base_num)

0.7982745751033534


In [87]:
for x in [1,2,3,4,5, 10, 25, 50, 100]:
    m1=xgb.XGBClassifier(max_depth=x)
    m1.fit(x_train, y_train)
    m1_num=roc_auc_score(y_val, m1.predict_proba(x_val)[:,1])
    print(f"Model AUC (when max_depth={x}): {m1_num}")

Model AUC (when max_depth=1): 0.8130885392742306
Model AUC (when max_depth=2): 0.8161783704639411
Model AUC (when max_depth=3): 0.8130670073495636
Model AUC (when max_depth=4): 0.8119724678456592
Model AUC (when max_depth=5): 0.7989205328433624
Model AUC (when max_depth=10): 0.7832704122645843
Model AUC (when max_depth=25): 0.7738609611851173
Model AUC (when max_depth=50): 0.7718584921910887
Model AUC (when max_depth=100): 0.7718584921910887


In [88]:
for x in [10, 15, 20, 25, 30, 35, 50, 100]:
    m2=xgb.XGBClassifier(n_estimators=x)
    m2.fit(x_train, y_train)
    m2_num=roc_auc_score(y_val, m2.predict_proba(x_val)[:,1])
    print(f"Model AUC (when n_estimators={x}): {m2_num}")

Model AUC (when n_estimators=10): 0.8065069476343592
Model AUC (when n_estimators=15): 0.8074148771244832
Model AUC (when n_estimators=20): 0.8075907211759301
Model AUC (when n_estimators=25): 0.8079352319706017
Model AUC (when n_estimators=30): 0.8075799552135967
Model AUC (when n_estimators=35): 0.8060583658704639
Model AUC (when n_estimators=50): 0.8026419671566375
Model AUC (when n_estimators=100): 0.7982745751033534


In [89]:
for x in [0.001, 0.01, 0.05, 0.065, 0.075, 0.085, 0.1, 0.25, 0.5, 1]:
    m3=xgb.XGBClassifier(learning_rate=x, n_estimators=25)
    m3.fit(x_train, y_train)
    m3_num=roc_auc_score(y_val, m3.predict_proba(x_val)[:,1])
    print(f"Model AUC (when learning_rate={x}): {m3_num}")

Model AUC (when learning_rate=0.001): 0.7844116042719339
Model AUC (when learning_rate=0.01): 0.7994085898024804
Model AUC (when learning_rate=0.05): 0.8036037264584289
Model AUC (when learning_rate=0.065): 0.8108671623794212
Model AUC (when learning_rate=0.075): 0.8115382406982086
Model AUC (when learning_rate=0.085): 0.8115382406982086
Model AUC (when learning_rate=0.1): 0.8112116731740927
Model AUC (when learning_rate=0.25): 0.8108492191088653
Model AUC (when learning_rate=0.5): 0.7933258210840607
Model AUC (when learning_rate=1): 0.7863925413412954


In [90]:
for x in [1,2,3,4,5,6, 7,10, 12,]:
    m4=xgb.XGBClassifier(max_depth=x, n_estimators=25, learning_rate=0.075)
    m4.fit(x_train, y_train)
    m4_num=roc_auc_score(y_val, m4.predict_proba(x_val)[:,1])
    print(f"Model AUC (when max_depth={x}): {m4_num}")

Model AUC (when max_depth=1): 0.7743023656407902
Model AUC (when max_depth=2): 0.8069914159393661
Model AUC (when max_depth=3): 0.8107702687184198
Model AUC (when max_depth=4): 0.8093850482315113
Model AUC (when max_depth=5): 0.810088424437299
Model AUC (when max_depth=6): 0.8115382406982086
Model AUC (when max_depth=7): 0.8083084519981627
Model AUC (when max_depth=10): 0.7973487023426733
Model AUC (when max_depth=12): 0.784547973128158


In [91]:
m5=xgb.XGBClassifier(max_depth=6, n_estimators=25, learning_rate=0.075)
m5.fit(x_train, y_train)
m5_num=roc_auc_score(y_val, m5.predict_proba(x_val)[:,1])
print(f"Model AUC (when max_depth=6, n_estimators=25, learning_rate=0.075): {m5_num}")

Model AUC (when max_depth=6, n_estimators=25, learning_rate=0.075): 0.8115382406982086


In [92]:
grid = {
    "max_depth": [ 1,3,6,10],
    "learning_rate": [0.005, 0.0075, 0.01, 0.025, 0.05],
    "n_estimators": [100, 500, 750, 1000],
    "reg_alpha": [0, 0.1, 1], 
    "reg_lambda": [1, 5, 10]
}

search = GridSearchCV(
    xgb.XGBClassifier(subsample=0.8, colsample_bytree=0.8, random_state=42),
    grid, scoring="roc_auc", n_jobs=-1, verbose=1, cv=5,
)

search.fit(x_train, y_train)
search_num=roc_auc_score(y_val, search.predict_proba(x_val)[:,1])
print(f"Search AUC: {search_num}")
print(f"Best Parameters: {search.best_params_}")
print(f"Best Cross-Validation Score: {search.best_score_}")
best_model = search.best_estimator_
print(f"Best Model AUC: {roc_auc_score(y_val, best_model.predict_proba(x_val)[:,1])}")

Fitting 5 folds for each of 720 candidates, totalling 3600 fits
Search AUC: 0.8115597726228755
Best Parameters: {'learning_rate': 0.025, 'max_depth': 1, 'n_estimators': 1000, 'reg_alpha': 0, 'reg_lambda': 1}
Best Cross-Validation Score: 0.8253240140076723
Best Model AUC: 0.8115597726228755


In [101]:
start = time.time()
final = xgb.XGBClassifier(max_depth=3, learning_rate=0.0075, n_estimators=1000,
                          subsample=0.8, colsample_bytree=0.8, random_state=42)
final.fit(x_train, y_train, eval_set=[(x_train, y_train), (x_val, y_val)], verbose=False)
xgb_time = time.time() - start
final_num = roc_auc_score(y_test, final.predict_proba(x_test)[:, 1])
print(f"Final Model AUC: {final_num}")
print(f"XGBoost training time: {xgb_time:.2f} seconds") 

Final Model AUC: 0.8321243121754631
XGBoost training time: 4.45 seconds


In [102]:
joblib.dump(final, '../models/xgb_model.joblib')

['../models/xgb_model.joblib']